# Data quality scan: buildings with `HtTot` (Oslo + Drammen)

Goal: decide, from facts, which buildings go into the silver/gold layers for predicting `HtTot` from weather + time.
Run the cells top to bottom. Each step ends with a **Decision** note so you can write down your thresholds before preprocessing.

Only the header rows are read in steps 1-2. From step 3 the data is loaded building by building.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

ROOT = Path("..") / "datasets"
SOURCES = {
    "oslo": ROOT / "COFACTOR-SBHUB-Oslo" / "Dataset_buildings_csv_files",
    "drammen": ROOT / "Cofactor_Drammen_Buildings_45_v3",
}
WEATHER = ["Tout", "SolGlob", "WindSpd", "WindDir", "RH"]

## 1. Read only the metadata + column row of every file
Line 1 is `Header_line;N`, and `N` is the line number of the column row. Lines 2 up to the blank line are `key;value` metadata.

In [ ]:
def read_head(path):
    """Return (meta dict, column list, header line number) without reading the data rows."""
    meta, cols = {}, None
    with open(path, encoding="utf-8", errors="replace") as f:
        n = int(f.readline().strip().split(";")[1])
        for i in range(2, n + 1):
            line = f.readline().rstrip("\r\n")
            if i == n:
                cols = line.split(";")
            elif ";" in line and line.split(";")[0]:
                k, v = line.split(";", 1)
                meta[k] = v
    return meta, cols, n

rows = []
for city, folder in SOURCES.items():
    for p in sorted(folder.glob("*.txt")):
        meta, cols, n = read_head(p)
        rows.append({"city": city, "file": p.name, "path": p, "header_line": n,
                     "cols": cols, **meta})
files = pd.DataFrame(rows)
try:
    files["building_id"] = files["building_id"].astype(int)
except ValueError as e:
    print("building_id is not a clean integer:", e)
    bad = files[~files["building_id"].astype(str).str.fullmatch(r"\d+")]
    print(f"\n{len(bad)} file(s) with a bad building_id:")
    print(bad[["city", "file", "building_id", "header_line"]].to_string())
    print("\nFirst rows of the dataframe:")
    print(files.head().to_string())
print(files.groupby("city").size())
files[["city", "file", "building_id", "header_line", "building_category", "year_of_construction", "floor_area"]].head()

## 2. Which buildings have `HtTot`?
Check: do all rows have a unique `building_id`, and how many buildings per city have `HtTot` in the header?

In [ ]:
files["has_HtTot"] = files["cols"].apply(lambda c: "HtTot" in c)
files["has_HtDH"] = files["cols"].apply(lambda c: "HtDH" in c)
print("duplicate building_id:", files["building_id"].duplicated().sum())
print(files.groupby("city")[["has_HtTot", "has_HtDH"]].sum())
print("total with HtTot:", files["has_HtTot"].sum())

target = files[files["has_HtTot"]].reset_index(drop=True)
print("\nbuilding_category among target buildings:")
print(target.groupby(["city", "building_category"]).size())

**Decision:** you counted about 37, my header scan gives 38 (33 Oslo + 5 Drammen). Step 4 will show whether one of them has an empty `HtTot`.

## 3. Common columns across the target buildings
Weather columns are expected everywhere. Check which columns are present in every target building and which are only in some.

In [ ]:
from collections import Counter
cnt = Counter(c for cols in target["cols"] for c in cols)
common = pd.Series(cnt).sort_values(ascending=False).rename("n_buildings").to_frame()
common["share"] = (common["n_buildings"] / len(target)).round(2)
print(common)
print("\nin every target building:", common.index[common["n_buildings"] == len(target)].tolist())

**Decision:** `RH` is expected to be missing in Drammen. Choose: drop `RH`, or keep it and fill/skip for Drammen.

## 4. Load each target building and build a per-building quality table
Timestamps are fixed UTC+1 all year (no DST), so the offset is stripped and times are kept naive, like in the earlier check notebook.
This cell reads the data of the target buildings only, one at a time, and keeps just a summary.

In [ ]:
def load(row):
    df = pd.read_csv(row["path"], sep=";", skiprows=row["header_line"] - 1)
    raw_ts = df["TimeStamp"]
    offsets = raw_ts.str[-5:].unique().tolist()
    df["ts"] = pd.to_datetime(raw_ts.str[:19])
    df = df.set_index("ts").drop(columns="TimeStamp").sort_index()
    return df, offsets

def longest_run(mask):
    """Length of the longest run of True in a boolean Series."""
    grp = (~mask).cumsum()
    return int(mask.groupby(grp).sum().max()) if mask.any() else 0

summ = []
for _, r in target.iterrows():
    df, offsets = load(r)
    idx = df.index
    full = pd.date_range(idx.min(), idx.max(), freq="h")
    h = df["HtTot"]
    flat = h.eq(h.shift()) & h.gt(0)
    summ.append({
        "city": r["city"], "id": r["building_id"], "cat": r["building_category"],
        "rows": len(df), "start": idx.min().date(), "end": idx.max().date(),
        "years": round((idx.max() - idx.min()).days / 365.25, 2),
        "missing_hours": len(full.difference(idx)), "dup_ts": int(idx.duplicated().sum()),
        "offsets": ",".join(offsets),
        "Ht_nan_%": round(h.isna().mean() * 100, 1),
        "Ht_zero_%": round((h == 0).mean() * 100, 1),
        "Ht_neg": int((h < 0).sum()),
        "Ht_flat_%": round(flat.mean() * 100, 1),
        "Ht_longest_nan_h": longest_run(h.isna()),
        "Ht_max": h.max(),
        "Tout_nan_%": round(df["Tout"].isna().mean() * 100, 1),
        "Tout_sentinel": int((df["Tout"] < -100).sum()),
        "Sol_neg": int((df["SolGlob"] < 0).sum()),
        "RH_present": "RH" in df.columns,
        "HtTot==HtDH": bool("HtDH" in df.columns and (df["HtTot"].dropna() == df["HtDH"].reindex(df["HtTot"].dropna().index)).mean() > 0.99),
    })
quality = pd.DataFrame(summ).sort_values(["city", "id"]).reset_index(drop=True)
quality

Read the table column by column:
- `rows`, `years`: how much data per building.
- `missing_hours`, `dup_ts`: gaps and duplicate timestamps in the time index.
- `offsets`: should be only `+0100`. Anything else means the fixed-UTC+1 assumption is wrong for that file.
- `Ht_nan_%`, `Ht_longest_nan_h`: how empty the target is, and the longest hole.
- `Ht_zero_%`, `Ht_neg`, `Ht_flat_%`: suspicious target values (zeros, negatives, the same value repeated).
- `Tout_sentinel`, `Sol_neg`: known weather problems (-273.15 sentinel, negative solar).
- `HtTot==HtDH`: whether the target is just a copy of district heating (so `HtDH` must never be a feature).

## 5. Summary statistics over all target buildings

In [ ]:
print(quality.describe().T[["min", "25%", "50%", "75%", "max"]].round(2))
print("\nbuildings with non-+0100 offsets:", (quality["offsets"] != "+0100").sum())
print("buildings with any duplicate timestamps:", (quality["dup_ts"] > 0).sum())
print("buildings where HtTot is a copy of HtDH:", quality["HtTot==HtDH"].sum())
print("total rows:", quality["rows"].sum())

## 6. Flag problem buildings
Change the thresholds to your own choices, then look at which buildings fall out.

In [ ]:
# EDIT THESE after reading steps 4-5
MAX_HT_NAN_PCT = 20       # target mostly empty
MAX_LONGEST_NAN_H = 24 * 30  # one hole longer than this many hours
MAX_HT_ZERO_PCT = 20      # target mostly zero
MAX_HT_FLAT_PCT = 20      # target mostly repeated values
MIN_YEARS = 1.0           # at least a full seasonal cycle

q = quality.copy()
q["flag_nan"] = q["Ht_nan_%"] > MAX_HT_NAN_PCT
q["flag_hole"] = q["Ht_longest_nan_h"] > MAX_LONGEST_NAN_H
q["flag_zero"] = q["Ht_zero_%"] > MAX_HT_ZERO_PCT
q["flag_flat"] = q["Ht_flat_%"] > MAX_HT_FLAT_PCT
q["flag_short"] = q["years"] < MIN_YEARS
flag_cols = [c for c in q.columns if c.startswith("flag_")]
q["n_flags"] = q[flag_cols].sum(axis=1)
print("flagged:", (q["n_flags"] > 0).sum(), "of", len(q))
q[q["n_flags"] > 0][["city", "id", "years", "Ht_nan_%", "Ht_longest_nan_h", "Ht_zero_%", "Ht_flat_%"] + flag_cols]

## 7. Look at the target by eye
Monthly total `HtTot` (MWh) per building. A building should show high winter and low summer values. Odd shapes (flat, all zero, big jumps) deserve a closer look.

In [ ]:
monthly = {}
for _, r in target.iterrows():
    df, _ = load(r)
    monthly[f'{r["city"]}-{r["building_id"]}'] = (df["HtTot"].resample("MS").sum(min_count=1) / 1e6).round(1)
pd.DataFrame(monthly).T

## 8. Relationship to weather
Correlation of `HtTot` with outdoor temperature per building. A strongly negative value is expected (colder, more heat). A value near zero suggests the building is hard to predict from weather alone or the data is broken.

In [ ]:
corr = {}
for _, r in target.iterrows():
    df, _ = load(r)
    d = df[df["Tout"] > -100]
    corr[f'{r["city"]}-{r["building_id"]}'] = d["HtTot"].corr(d["Tout"])
pd.Series(corr, name="corr(HtTot, Tout)").round(2).sort_values()

## 9. Write down your decisions
Fill these in before the preprocessing step.

- Buildings to exclude (id and reason):
- `RH`: drop / keep:
- Gap rule for weather (interpolate up to N hours?):
- Gap rule for `HtTot` (never fill, drop those rows):
- Handling of `HtTot == 0` and flat stretches:
- Train / validation / test split by building (which ids go where):